In [ ]:
import pandas as pd

df = pd.read_csv("ds_jobs.csv")
#standardizing job titles
df["title"] = df["title"].str.lower()
df["title"] = df["title"].str.replace("sr.", "senior")
df["title"] = df["title"].str.replace(r"\b(i|ii|iii)\b", "", regex=True)
#grouping by job type
def job_type(title):
    title = str(title).lower()
    if "data scientist" in title or "data science" in title:
        return "Data Scientist"
    elif "data engineer" in title or "data engineering" in title:
        return "Data Engineer"
    elif "data analyst" in title or "data analysis" in title or "analyst" in title:
        return "Data Analyst"
    elif "machine learning scientist" in title or "machine learning science" in title:
        return "ML Scientist"
    elif "machine learning engineer" in title or "ml engineer" in title or "ml engineering" in title or "machine learning engineering" in title:
        return "ML Engineer"
    elif "ai engineer" in title or "ai engineering" in title:
        return "AI Engineer"
    elif "ai/ml engineer" in title or "ai/ml engineering" in title:
        return "AI/ML Engineer"
    elif "deep learning" in title:
        return "Deep Learning Engineer"
    elif "big data engineer" in title or "big data engineering" in title:
        return "Big Data Engineer"
    elif "analytics engineer" in title or "analytics engineering" in title:
        return "Analytics Engineer"
    elif "business analyst" in title or "business analysis" in title:
        return "Business Analyst"
    elif "bi analyst" in title or "business intelligence" in title or "bi analysis" in title or "business intelligence analysis" in title:
        return "Business Intelligence Analyst"
    elif "software engineer" in title or "software engineering" in title:
        return "Software Engineer"
    elif "data manager" in title or "manager" in title or "data management" in title or "management" in title:
        return "Data Manager"
    elif "cloud" in title:
        return "Cloud Engineer"
    elif "database" in title or "db" in title:
        return "Database Engineer"
    else:
        return "Other"
df["job_type"] = df["title"].apply(job_type)
#extract years of experience from description
df["description"] = df["description"].str.lower()
def extract_experience(description, title = ""):
    if pd.isna(description):
        return None
    description = description.lower()
    title = str(title).lower()
    #handle a range like 5-7 years
    range_match = re.search(r'\b(\d{1,2})\s*[-–]\s*(\d{1,2})\s*(?:years?|yrs?)\b', description)
    #average for ranges
    if range_match:
        low, high = int(range_match.group(1)), int(range_match.group(2))
        value = (low + high) / 2
    else:
        single_match = re.search(r'\b(\d{1,2})\+?\s*(?:years?|yrs?)\b', description)
        if single_match:
            value = float(single_match.group(1))
        else:
            return None
    return value
df["years_experience"] = df["description"].apply(extract_experience)
#clean locations
df["location"] = df["location"].str.strip()
df[["part1", "part2", "part3"]] = df["location"].str.split(",", expand=True)
df["part1"] = df["part1"].str.strip()
df["part2"] = df["part2"].str.strip()
df["part3"] = df["part3"].str.strip()
df["state"] = df["part2"].fillna(df["part1"])
df.loc[df["state"] == "US", "state"] = None
df.loc[df["state"] == "Remote", "state"] = None
#removes remote jobs from map
df.loc[df["is_remote"] == True, "state"] = None
jobs_per_state = (
    df.groupby("state")
      .size()
      .reset_index(name="job_count")
)
#clean company size
df["company_size"] = df["company_num_employees"].str.replace("employees", "", case=False).str.strip()
def company_size_clean(size):
    if pd.isna(size) or str(size).lower() == "decline to state":
        return None
    #remove commas
    size = str(size).lower().replace(",", "")
    if "-" in size:
        parts = size.split("-")
        low = int(parts[0])
        high = int(parts[1])
        return (low + high) / 2
    if "to" in size:
        parts = size.split(" to ")
        low = int(parts[0].strip())
        high = int(parts[1].strip())
        return (low + high) / 2
    if "+" in size:
        return int(size.replace("+", ""))
    try:
        return int(size)
    except:
        return None
df["company_size_numeric"] = df["company_size"].apply(company_size_clean)

In [67]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8935 entries, 0 to 8934
Data columns (total 42 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0.1           8935 non-null   int64  
 1   Unnamed: 0             8935 non-null   int64  
 2   id                     8935 non-null   str    
 3   site                   8935 non-null   str    
 4   job_url                8935 non-null   str    
 5   job_url_direct         8045 non-null   str    
 6   title                  8935 non-null   str    
 7   company                8776 non-null   str    
 8   location               8924 non-null   str    
 9   job_type               8935 non-null   str    
 10  date_posted            8935 non-null   str    
 11  salary_source          7940 non-null   str    
 12  interval               8248 non-null   str    
 13  min_amount             7940 non-null   float64
 14  max_amount             7940 non-null   float64
 15  currency       

In [69]:
# remove unnecessary columns
df = df.drop([
    "job_level", "job_function", "Unnamed: 0.1", "Unnamed: 0", "id", "job_url", "job_url_direct", "logo_photo_url", "banner_photo_url", "ceo_name", "ceo_photo_url",
    "salary_source", "interval", "emails", "company_url", "company_url_direct", "listing_type"], axis=1)

In [70]:
df.head()

,site,title,company,location,job_type,date_posted,min_amount,max_amount,currency,is_remote,...,company_description,mean_salary,cleaned_description,years_experience,part1,part2,part3,state,company_size,company_size_numeric
0,indeed,senior data analyst,General Motors,"Warren, MI, US",Data Analyst,2024-10-10 00:00:00,100400.0,127129.0,USD,0.0,...,"We see a future with Zero Crashes, Zero Emissi...",113764.5,**job description** ------------------- we are...,7.0,Warren,MI,US,MI,"10,000+",10000.0
1,indeed,senior ai/ml engineer - core,Thornton Tomasetti,"New York, NY, US",ML Engineer,2024-10-10 00:00:00,84000.0,94500.0,USD,0.0,...,NaN,89250.0,thornton tomasetti applies engineering and sci...,NaN,New York,NY,US,NY,NaN,NaN
2,indeed,ai & data analytics manager,Lionbridge,"Boise, ID, US",Data Manager,2024-10-10 00:00:00,105024.0,132984.0,USD,1.0,...,Breaking barriers. Building bridges. The world...,119004.0,**ai & data analytics manager** at lionbridge ...,NaN,Boise,ID,US,NaN,"5,001 to 10,000",7500.5
3,indeed,postdoctoral fellowships - department of data ...,Dana-Farber Cancer Institute,"Boston, MA, US",Data Scientist,2024-10-10 00:00:00,56326.0,71321.0,USD,0.0,...,The Dana-Farber Cancer Institute fights cancer...,63823.5,**job ref:** 41777 **location:** 450 brookline...,NaN,Boston,MA,US,MA,"1,001 to 5,000",3000.5
4,indeed,"tech lead manager, data acquisition",OpenAI,"San Francisco, CA, US",Data Manager,2024-10-10 00:00:00,310000.0,385000.0,USD,0.0,...,NaN,347500.0,"**technical lead manager, data acquisition** *...",7.0,San Francisco,CA,US,CA,51 to 200,125.5


In [71]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8935 entries, 0 to 8934
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   site                   8935 non-null   str    
 1   title                  8935 non-null   str    
 2   company                8776 non-null   str    
 3   location               8924 non-null   str    
 4   job_type               8935 non-null   str    
 5   date_posted            8935 non-null   str    
 6   min_amount             7940 non-null   float64
 7   max_amount             7940 non-null   float64
 8   currency               8248 non-null   str    
 9   is_remote              8115 non-null   float64
 10  company_industry       1875 non-null   str    
 11  description            8935 non-null   str    
 12  company_addresses      6159 non-null   str    
 13  company_num_employees  6230 non-null   str    
 14  company_revenue        5593 non-null   str    
 15  company_descrip

In [72]:
# to csv
df.to_csv("data/ds_jobs.csv", index=False)

In [60]:
# select distinct for company_num_employees
df["company_num_employees"].unique()

<ArrowStringArray>
[         '10,000+',                nan,  '5,001 to 10,000',
   '1,001 to 5,000',        '51 to 200',       '201 to 500',
 'Decline to state',          '2 to 10',         '11 to 50',
     '501 to 1,000',          '1 to 50',                '1']
Length: 12, dtype: str

In [61]:
df_salary_per_company_size = df[["title", "company_num_employees", "mean_salary", "location"]]

In [62]:
# remove rows with missing values for company_num_employees
df_salary_per_company_size = df_salary_per_company_size.dropna(subset=["company_num_employees"])

# remove rows with "Decline to state" for company_num_employees
df_salary_per_company_size = df_salary_per_company_size[df_salary_per_company_size["company_num_employees"] != "Decline to state"]

In [63]:
# combine "1", "1 to 50", "2 to 10", "11 to 50" into "1 to 50"
df_salary_per_company_size.loc[df_salary_per_company_size["company_num_employees"].isin(["1", "1 to 50", "2 to 10", "11 to 50"]), "company_num_employees"] = "1 to 50"

df_salary_per_company_size["company_num_employees"].unique()

<ArrowStringArray>
[        '10,000+', '5,001 to 10,000',  '1,001 to 5,000',       '51 to 200',
      '201 to 500',         '1 to 50',    '501 to 1,000']
Length: 7, dtype: str

In [64]:
size_order = [
    '1 to 50',
    '51 to 200',
    '201 to 500',
    '501 to 1,000',
    '1,001 to 5,000',
    '5,001 to 10,000',
    '10,000+'
]

df_salary_per_company_size['company_num_employees'] = pd.Categorical(
    df_salary_per_company_size['company_num_employees'],
    categories=size_order,
    ordered=True
)

df_salary_per_company_size = df_salary_per_company_size.sort_values('company_num_employees')

In [65]:
import altair as alt
alt.renderers.enable('default')
alt.data_transformers.enable('vegafusion')

chart = alt.Chart(df_salary_per_company_size).mark_boxplot(
    size=40,
    ticks=True,
    outliers=True
).encode(
    x=alt.X('company_num_employees:O',
            title='Company Size',
            sort=size_order),
    y=alt.Y('mean_salary:Q',
            title='Salary'),
    tooltip=[
        'company_num_employees',
        alt.Tooltip('mean_salary:Q', format='.0f', title='Salary'),
    ]
).properties(
    width=600,
    height=400,
    title='Salary by Company Size'
).configure_axis(
    labelAngle=0
).configure_title(
    fontSize=16
)

chart

alt.Chart(...)